In [3]:
import os
import pandas as pd
import numpy as np


# STEP 1: CREATE DIRECTORY STRUCTURE
# create folder in computer
os.makedirs('data/raw', exist_ok=True)
os.makedirs('outputs', exist_ok=True)

print("Directories 'data/raw/' and 'outputs/' are ready.")

# STEP 2: GENERATE AND SAVE MOCK DATA
# Set seed for reproducibility
np.random.seed(42)

# 1. Generate Mock Google Search Console (GSC) Data
gsc_data = {
    'page_url': [f'https://mysite.com/blog/page-{i}' for i in range(1, 101)],
    'query': [f'keyword choice {i}' for i in range(1, 101)],
    'clicks_current': np.random.randint(50, 5000, size=100),
    'clicks_previous': np.random.randint(60, 6000, size=100),
    'impressions': np.random.randint(1000, 100000, size=100),
    'avg_position': np.random.uniform(1.0, 30.0, size=100)
}
df_gsc_raw = pd.DataFrame(gsc_data)
# Save raw GSC data to the folder
df_gsc_raw.to_csv('data/raw/gsc_raw_data.csv', index=False)

# 2. Generate Mock Google Analytics 4 (GA4) Data
ga4_data = {
    'landing_page': [f'https://mysite.com/blog/page-{i}' for i in range(1, 95)], 
    'sessions': np.random.randint(100, 8000, size=94),
    'conversions': np.random.randint(0, 150, size=94),
    'revenue': np.random.uniform(0.0, 5000.0, size=94).round(2)
}
df_ga4_raw = pd.DataFrame(ga4_data)
# Save raw GA4 data to the folder
df_ga4_raw.to_csv('data/raw/ga4_raw_data.csv', index=False)

print("Success: Raw CSV files generated and saved to 'data/raw/'.\n")


# STEP 3: READ FILES BACK IN (The ETL Process)


df_gsc = pd.read_csv('data/raw/gsc_raw_data.csv')
df_ga4 = pd.read_csv('data/raw/ga4_raw_data.csv')

# Standardize URLs for a clean merge
df_gsc['page_url'] = df_gsc['page_url'].str.strip().str.lower()
df_ga4['landing_page'] = df_ga4['landing_page'].str.strip().str.lower()

# Merge Datasets
df_merged = pd.merge(df_gsc, df_ga4, left_on='page_url', right_on='landing_page', how='inner')
df_merged.drop(columns=['landing_page'], inplace=True)

# STEP 4: CALCULATE SEO METRICS & SEGMENT

df_merged['ctr'] = (df_merged['clicks_current'] / df_merged['impressions']).round(4)
df_merged['conversion_rate'] = (df_merged['conversions'] / df_merged['sessions']).round(4)
df_merged['traffic_decay_pct'] = ((df_merged['clicks_current'] - df_merged['clicks_previous']) / df_merged['clicks_previous']).round(4)

def assign_seo_action(row):
    if row['traffic_decay_pct'] <= -0.15:
        return 'Urgent Content Refresh (Traffic Decay)'
    elif 7 <= row['avg_position'] <= 15:
        return 'Optimize Metadata / On-Page (Striking Distance)'
    elif row['clicks_current'] > 2000 and row['conversion_rate'] < 0.01:
        return 'Fix UX / CRO Opportunity'
    else:
        return 'Monitor / Maintain'

df_merged['recommended_action'] = df_merged.apply(assign_seo_action, axis=1)


# STEP 5: EXPORT MASTER DATA FOR DASHBOARD

# Save the final cleaned, engineered dataset into the outputs folder
df_merged.to_csv('outputs/seo_master_dashboard_data.csv', index=False)

print("Success: Cleaned master dataset saved to 'outputs/seo_master_dashboard_data.csv'.")

Directories 'data/raw/' and 'outputs/' are ready.
Success: Raw CSV files generated and saved to 'data/raw/'.

Success: Cleaned master dataset saved to 'outputs/seo_master_dashboard_data.csv'.
